<a href="https://colab.research.google.com/github/madaam99/EnergyGermany/blob/main/SMARD_DataAcquisition_Energy_Germany_Monthly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Getting the data

In [1]:
import requests
from datetime import datetime, timedelta
import os

In [2]:
def smard_download(payload, filename):
    """
    Downloads SMARD CSV data using the given payload and saves it to filename.
    """
    url = "https://www.smard.de/nip-download-manager/nip/download/market-data"

    headers = {
        "Content-Type": "application/json",
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.post(url, json=payload, headers=headers)

    if response.status_code != 200:
        raise RuntimeError(f"SMARD returned status {response.status_code}")

    text = response.text

    # SMARD sometimes returns a CSV that only contains "Keine Daten"
    if "Keine Daten" in text:
        print(f"No data available for: {filename}")
    else:
        print(f"Download OK for: {filename}")

    with open(filename, "wb") as f:
        f.write(response.content)

In [3]:
today = datetime.today()
five_years_ago = today - timedelta(days=5*365)

timestamp_to = int(today.timestamp() * 1000)
timestamp_from = int(five_years_ago.timestamp() * 1000)

payload_installed_energy = {
    "request_form": [
        {
            "format": "CSV",
            "moduleIds": [
                3004073,3004076,3004072,3004074,3004075,
                3000186,3000188,3000189,3000194,3000198,
                3003792,3000207
            ],
            "region": "DE",
            "timestamp_from": timestamp_from,
            "timestamp_to": timestamp_to,
            "type": "discrete",
            "language": "de",
            "resolution": "month"
        }
    ]
}

payload_energy_production = {
    "request_form": [
        {
            "format": "CSV",
            "moduleIds": [
                1001224,1004066,1004067,1004068,
                1001223,1004069,1004071,1004070,
                1001226,1001228,1001227,1001225
            ],
            "region": "DE",
            "timestamp_from": timestamp_from,
            "timestamp_to": timestamp_to,
            "type": "discrete",
            "language": "de",
            "resolution": "month"
        }
    ]
}

payload_energy_consumption = {
    "request_form": [
        {
            "format": "CSV",
            "moduleIds": [
                5000410,5004387,5005140,5004359
            ],
            "region": "DE",
            "timestamp_from": timestamp_from,
            "timestamp_to": timestamp_to,
            "type": "discrete",
            "language": "de",
            "resolution": "month"
        }
    ]
}

smard_download(payload_installed_energy, "installed_energy.csv")
smard_download(payload_energy_production, "energy_production_monthly.csv")
smard_download(payload_energy_consumption, "energy_consumption_monthly.csv")

print("All downloads completed.")

Download OK for: installed_energy.csv
Download OK for: energy_production_monthly.csv
Download OK for: energy_consumption_monthly.csv
All downloads completed.


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

Installed Energy Production by Sources

In [5]:
instEnergy = pd.read_csv('/content/installed_energy.csv', delimiter=";")
instEnergy.head()

,Datum von,Datum bis,Biomasse [MW] Berechnete Auflösungen,Wasserkraft [MW] Berechnete Auflösungen,Wind Offshore [MW] Berechnete Auflösungen,Wind Onshore [MW] Berechnete Auflösungen,Photovoltaik [MW] Berechnete Auflösungen,Sonstige Erneuerbare [MW] Berechnete Auflösungen,Kernenergie [MW] Berechnete Auflösungen,Braunkohle [MW] Berechnete Auflösungen,Steinkohle [MW] Berechnete Auflösungen,Erdgas [MW] Berechnete Auflösungen,Pumpspeicher [MW] Berechnete Auflösungen,Sonstige Konventionelle [MW] Berechnete Auflösungen
0,01.01.2021,01.02.2021,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-
1,01.02.2021,01.03.2021,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-
2,01.03.2021,01.04.2021,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-
3,01.04.2021,01.05.2021,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-
4,01.05.2021,01.06.2021,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-


In [6]:
instEnergy['Datum von'] = pd.to_datetime(instEnergy['Datum von'], format="%d.%m.%Y")
instEnergy['Datum bis'] = pd.to_datetime(instEnergy["Datum bis"], format="%d.%m.%Y")

In [7]:
instEnergy = instEnergy.rename(columns=lambda x: x.replace("Berechnete Auflösungen", "").strip())
instEnergy.head()

,Datum von,Datum bis,Biomasse [MW],Wasserkraft [MW],Wind Offshore [MW],Wind Onshore [MW],Photovoltaik [MW],Sonstige Erneuerbare [MW],Kernenergie [MW],Braunkohle [MW],Steinkohle [MW],Erdgas [MW],Pumpspeicher [MW],Sonstige Konventionelle [MW]
0,2021-01-01,2021-02-01,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-
1,2021-02-01,2021-03-01,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-
2,2021-03-01,2021-04-01,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-
3,2021-04-01,2021-05-01,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-
4,2021-05-01,2021-06-01,"8.204,00",-,-,"54.345,00","50.410,00",-,-,-,-,"31.942,00",-,-


In [8]:
instEnergy = instEnergy.replace("-", np.nan).fillna(0)
instEnergy.head()

,Datum von,Datum bis,Biomasse [MW],Wasserkraft [MW],Wind Offshore [MW],Wind Onshore [MW],Photovoltaik [MW],Sonstige Erneuerbare [MW],Kernenergie [MW],Braunkohle [MW],Steinkohle [MW],Erdgas [MW],Pumpspeicher [MW],Sonstige Konventionelle [MW]
0,2021-01-01,2021-02-01,"8.204,00",0,0,"54.345,00","50.410,00",0,0.0,0,0,"31.942,00",0,0
1,2021-02-01,2021-03-01,"8.204,00",0,0,"54.345,00","50.410,00",0,0.0,0,0,"31.942,00",0,0
2,2021-03-01,2021-04-01,"8.204,00",0,0,"54.345,00","50.410,00",0,0.0,0,0,"31.942,00",0,0
3,2021-04-01,2021-05-01,"8.204,00",0,0,"54.345,00","50.410,00",0,0.0,0,0,"31.942,00",0,0
4,2021-05-01,2021-06-01,"8.204,00",0,0,"54.345,00","50.410,00",0,0.0,0,0,"31.942,00",0,0


In [9]:
numCol = instEnergy.drop(columns=["Datum von", "Datum bis"]).columns

for col in numCol:
  instEnergy[col] = instEnergy[col].astype(str).apply(lambda x: x.replace(".", "").replace(",", ".")).astype(float)

instEnergy.dtypes

,0
Datum von,datetime64[ns]
Datum bis,datetime64[ns]
Biomasse [MW],float64
Wasserkraft [MW],float64
Wind Offshore [MW],float64
Wind Onshore [MW],float64
Photovoltaik [MW],float64
Sonstige Erneuerbare [MW],float64
Kernenergie [MW],float64
Braunkohle [MW],float64


In [10]:
instEnergy.insert(2, "Gesamtkapazität [MW]", instEnergy[numCol].sum(axis=1))
instEnergy.head()

,Datum von,Datum bis,Gesamtkapazität [MW],Biomasse [MW],Wasserkraft [MW],Wind Offshore [MW],Wind Onshore [MW],Photovoltaik [MW],Sonstige Erneuerbare [MW],Kernenergie [MW],Braunkohle [MW],Steinkohle [MW],Erdgas [MW],Pumpspeicher [MW],Sonstige Konventionelle [MW]
0,2021-01-01,2021-02-01,144901.0,8204.0,0.0,0.0,54345.0,50410.0,0.0,0.0,0.0,0.0,31942.0,0.0,0.0
1,2021-02-01,2021-03-01,144901.0,8204.0,0.0,0.0,54345.0,50410.0,0.0,0.0,0.0,0.0,31942.0,0.0,0.0
2,2021-03-01,2021-04-01,144901.0,8204.0,0.0,0.0,54345.0,50410.0,0.0,0.0,0.0,0.0,31942.0,0.0,0.0
3,2021-04-01,2021-05-01,144901.0,8204.0,0.0,0.0,54345.0,50410.0,0.0,0.0,0.0,0.0,31942.0,0.0,0.0
4,2021-05-01,2021-06-01,144901.0,8204.0,0.0,0.0,54345.0,50410.0,0.0,0.0,0.0,0.0,31942.0,0.0,0.0


In [11]:
instEnergy_mwh = instEnergy.copy()

instEnergy_mwh['Datum von'] = pd.to_datetime(instEnergy_mwh['Datum von'])
instEnergy_mwh['Datum bis'] = pd.to_datetime(instEnergy_mwh['Datum bis'])

instEnergy_mwh['Duration_Hours'] = (instEnergy_mwh['Datum bis'] - instEnergy_mwh['Datum von']).dt.total_seconds() / 3600

mw_columns = [col for col in instEnergy_mwh.columns if '[MW]' in col]

for col in mw_columns:
    instEnergy_mwh[col] = instEnergy_mwh[col] * instEnergy_mwh['Duration_Hours']

instEnergy_mwh.columns = [col.replace('[MW]', '[MWh]') if '[MW]' in col else col for col in instEnergy_mwh.columns]

instEnergy_mwh.drop(columns=['Duration_Hours'], inplace=True)

instEnergy_mwh.head()

,Datum von,Datum bis,Gesamtkapazität [MWh],Biomasse [MWh],Wasserkraft [MWh],Wind Offshore [MWh],Wind Onshore [MWh],Photovoltaik [MWh],Sonstige Erneuerbare [MWh],Kernenergie [MWh],Braunkohle [MWh],Steinkohle [MWh],Erdgas [MWh],Pumpspeicher [MWh],Sonstige Konventionelle [MWh]
0,2021-01-01,2021-02-01,107806344.0,6103776.0,0.0,0.0,40432680.0,37505040.0,0.0,0.0,0.0,0.0,23764848.0,0.0,0.0
1,2021-02-01,2021-03-01,97373472.0,5513088.0,0.0,0.0,36519840.0,33875520.0,0.0,0.0,0.0,0.0,21465024.0,0.0,0.0
2,2021-03-01,2021-04-01,107806344.0,6103776.0,0.0,0.0,40432680.0,37505040.0,0.0,0.0,0.0,0.0,23764848.0,0.0,0.0
3,2021-04-01,2021-05-01,104328720.0,5906880.0,0.0,0.0,39128400.0,36295200.0,0.0,0.0,0.0,0.0,22998240.0,0.0,0.0
4,2021-05-01,2021-06-01,107806344.0,6103776.0,0.0,0.0,40432680.0,37505040.0,0.0,0.0,0.0,0.0,23764848.0,0.0,0.0


In [12]:
instEnergy_mwh.to_csv("processedInstalledEnergy", index=False)

Produced Energy

In [13]:
erzeugung = pd.read_csv("/content/energy_production_monthly.csv", delimiter=";")
erzeugung.head()

,Datum von,Datum bis,Biomasse [MWh] Berechnete Auflösungen,Wasserkraft [MWh] Berechnete Auflösungen,Wind Offshore [MWh] Berechnete Auflösungen,Wind Onshore [MWh] Berechnete Auflösungen,Photovoltaik [MWh] Berechnete Auflösungen,Sonstige Erneuerbare [MWh] Berechnete Auflösungen,Kernenergie [MWh] Berechnete Auflösungen,Braunkohle [MWh] Berechnete Auflösungen,Steinkohle [MWh] Berechnete Auflösungen,Erdgas [MWh] Berechnete Auflösungen,Pumpspeicher [MWh] Berechnete Auflösungen,Sonstige Konventionelle [MWh] Berechnete Auflösungen
0,01.01.2021,01.02.2021,"3.390.654,25","992.324,50","2.263.541,25","9.190.255,25","600.445,75","154.603,00","5.925.379,00","10.260.437,50","5.111.958,25","7.292.294,75","797.113,50","1.190.998,50"
1,01.02.2021,01.03.2021,"2.972.072,25","1.067.784,75","2.743.174,50","8.451.315,00","1.966.106,50","130.328,00","5.343.534,75","7.686.818,75","3.350.323,25","6.358.551,00","680.576,50","1.019.811,25"
2,01.03.2021,01.04.2021,"3.321.780,50","1.096.885,50","2.356.745,00","9.052.932,75","4.084.762,00","145.858,25","5.502.068,00","7.844.513,50","3.502.914,00","6.603.361,75","770.015,25","1.170.678,50"
3,01.04.2021,01.05.2021,"3.215.143,75","1.028.506,25","1.775.864,25","7.709.987,75","5.411.394,25","113.368,50","5.145.542,75","7.086.525,50","3.452.755,50","5.587.178,00","656.823,00","1.077.606,50"
4,01.05.2021,01.06.2021,"3.348.917,75","1.547.943,00","1.287.883,50","8.792.241,50","5.861.740,75","109.287,50","5.277.141,25","5.706.664,75","2.393.617,25","3.589.174,50","571.920,75","1.111.334,50"


In [14]:
erzeugung["Datum von"] = pd.to_datetime(erzeugung["Datum von"], format="%d.%m.%Y")
erzeugung["Datum bis"] = pd.to_datetime(erzeugung["Datum bis"], format="%d.%m.%Y")

erzeugung = erzeugung.rename(columns=lambda x: x.replace("Berechnete Auflösungen", "").strip())

erzeugung = erzeugung.replace("-", np.nan).fillna(0)

numCols = erzeugung.columns.drop(["Datum von", "Datum bis"])

for col in numCols:
  erzeugung[col] = erzeugung[col].astype(str).str.replace(".", "").str.replace(",", ".").astype(float)

erzeugung = erzeugung.loc[~(erzeugung.iloc[:, 2:] == 0).all(axis=1)]

erzeugung.insert(2, "Gesamterzeugung [MWh]", erzeugung[numCols].sum(axis=1))

erzeugung.head()


,Datum von,Datum bis,Gesamterzeugung [MWh],Biomasse [MWh],Wasserkraft [MWh],Wind Offshore [MWh],Wind Onshore [MWh],Photovoltaik [MWh],Sonstige Erneuerbare [MWh],Kernenergie [MWh],Braunkohle [MWh],Steinkohle [MWh],Erdgas [MWh],Pumpspeicher [MWh],Sonstige Konventionelle [MWh]
0,2021-01-01,2021-02-01,47170005.5,3390654.25,992324.50,2263541.25,9190255.25,600445.75,154603.00,5925379.00,10260437.50,5111958.25,7292294.75,797113.50,1190998.50
1,2021-02-01,2021-03-01,41770396.5,2972072.25,1067784.75,2743174.50,8451315.00,1966106.50,130328.00,5343534.75,7686818.75,3350323.25,6358551.00,680576.50,1019811.25
2,2021-03-01,2021-04-01,45452515.0,3321780.50,1096885.50,2356745.00,9052932.75,4084762.00,145858.25,5502068.00,7844513.50,3502914.00,6603361.75,770015.25,1170678.50
3,2021-04-01,2021-05-01,42260696.0,3215143.75,1028506.25,1775864.25,7709987.75,5411394.25,113368.50,5145542.75,7086525.50,3452755.50,5587178.00,656823.00,1077606.50
4,2021-05-01,2021-06-01,39597867.0,3348917.75,1547943.00,1287883.50,8792241.50,5861740.75,109287.50,5277141.25,5706664.75,2393617.25,3589174.50,571920.75,1111334.50


In [15]:
erzeugung.to_csv("processedErzeugungMonthly", index=False)

Energy Consumption

In [16]:
verbrauch = pd.read_csv("/content/energy_consumption_monthly.csv", delimiter=";")
verbrauch.head()

,Datum von,Datum bis,Netzlast [MWh] Berechnete Auflösungen,Netzlast inkl. Pumpspeicher [MWh] Berechnete Auflösungen,Pumpspeicher [MWh] Berechnete Auflösungen,Residuallast [MWh] Berechnete Auflösungen
0,01.01.2021,01.02.2021,"45.631.444,00","46.635.089,25","1.003.645,25","33.577.061,88"
1,01.02.2021,01.03.2021,"41.758.874,75","42.590.296,50","831.421,75","28.598.278,75"
2,01.03.2021,01.04.2021,"44.880.865,50","45.849.448,50","968.583,00","29.386.425,75"
3,01.04.2021,01.05.2021,"41.055.628,50","41.927.799,75","872.171,25","26.158.382,25"
4,01.05.2021,01.06.2021,"40.235.351,00","41.063.334,00","827.983,00","24.293.485,25"


In [17]:
verbrauch['Datum von'] = pd.to_datetime(verbrauch['Datum von'], format="%d.%m.%Y")
verbrauch['Datum bis'] = pd.to_datetime(verbrauch['Datum bis'], format="%d.%m.%Y")

verbrauch = verbrauch.rename(columns=lambda x: x.replace("Berechnete Auflösungen", "").strip())

verbrauch = verbrauch.replace("-", np.nan).fillna(0)

cols = verbrauch.columns.drop(["Datum von", "Datum bis"])

for col in cols:
  verbrauch[col] = verbrauch[col].astype(str).str.replace(".", "").str.replace(",", ".").astype(float)

verbrauch = verbrauch.loc[~(verbrauch.iloc[:, 2:] == 0).all(axis=1)]

verbrauch.insert(2, "Gesamtlast [MWh]", verbrauch[["Netzlast [MWh]", "Pumpspeicher [MWh]", "Residuallast [MWh]"]].sum(axis=1))

verbrauch.head()

,Datum von,Datum bis,Gesamtlast [MWh],Netzlast [MWh],Netzlast inkl. Pumpspeicher [MWh],Pumpspeicher [MWh],Residuallast [MWh]
0,2021-01-01,2021-02-01,80212151.13,45631444.00,46635089.25,1003645.25,33577061.88
1,2021-02-01,2021-03-01,71188575.25,41758874.75,42590296.50,831421.75,28598278.75
2,2021-03-01,2021-04-01,75235874.25,44880865.50,45849448.50,968583.00,29386425.75
3,2021-04-01,2021-05-01,68086182.00,41055628.50,41927799.75,872171.25,26158382.25
4,2021-05-01,2021-06-01,65356819.25,40235351.00,41063334.00,827983.00,24293485.25


In [18]:
verbrauch.to_csv("processedVerbrauchMonthly", index=False)